In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_04_filtros_ordenacao")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 00:11:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv("/workspace/datasets/04_filtros_ordenacao/clientes.csv", header= True, inferSchema= True)

In [4]:
df.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- profissao: string (nullable = true)
 |-- renda_mensal: double (nullable = true)
 |-- sexo: string (nullable = true)
 |-- ativo: boolean (nullable = true)
 |-- data_cadastro: date (nullable = true)



In [5]:
df_maiores = (
    df.filter(F.col("idade") >= 35)
)

In [6]:
df_maiores

DataFrame[cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date]

In [7]:
df_maiores.show()

+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|      nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|         4| Cliente 4|   64|        Salvador|    BA|     Analista|     4253.53|   F|false|   2023-01-03|
|         5| Cliente 5|   60|        Salvador|    BA|   Engenheiro|     2710.92|   M|false|   2023-09-15|
|         7| Cliente 7|   46|Feira de Santana|    BA|    Professor|      6175.2|   M| true|   2023-12-15|
|        10|Cliente 10|   56|Feira de Santana|    BA|   Engenheiro|     8888.41|   F| true|   2024-02-22|
|        11|Cliente 11|   68|       São Paulo|    SP|Desenvolvedor|     5135.92|   M|false|   2024-04-26|
|        13|Cliente 13|   63|Feira de Santana|    BA|Desenvolvedor|     12740.2|   M|false|   2023-06-05|
|        17|Cliente 17|   62|       São Paulo|

In [8]:
df_cidade = (
    df.filter(
        ((F.col("cidade") == "Salvador") | (F.col("cidade") == "Recife"))
    )
)

In [9]:
df_clientes_qualificados = (
    df.filter(
        ((F.col("idade") >= 30) | (F.col("renda_mensal") >= 5000) | (F.col("ativo") == "true")) 
    )
)

In [10]:
df_clientes_qualificados.explain(True)

== Parsed Logical Plan ==
'Filter ((('idade >= 30) OR ('renda_mensal >= 5000)) OR ('ativo = true))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Filter (((idade#19 >= 30) OR (renda_mensal#23 >= cast(5000 as double))) OR (ativo#25 = cast(true as boolean)))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Filter (((idade#19 >= 30) OR (renda_mensal#23 >= 5000.0)) OR ativo#25)
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
*(1) Filter (((idade#19 >= 30) OR (renda_mensal#23 >= 5000.0)) OR ativo#25)
+

In [11]:
df_inativos = (
    df.filter(
        ~(F.col("ativo") == "true")
    )
)

In [12]:
df_fora_salvador = (
    df.filter(
        ~(F.col("cidade") == "Salvador")
    )
)

In [13]:
df_fora_salvador.explain(True)

== Parsed Logical Plan ==
'Filter NOT ('cidade = Salvador)
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Filter NOT (cidade#20 = Salvador)
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Filter (isnotnull(cidade#20) AND NOT (cidade#20 = Salvador))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
*(1) Filter (isnotnull(cidade#20) AND NOT (cidade#20 = Salvador))
+- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] Batche

In [14]:
df_ordenado = (
    df.orderBy(
        F.col("renda_mensal").desc()
    )
)

In [15]:
df_ordenado.show()

+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|       nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|       212|Cliente 212|   19|Feira de Santana|    BA|Desenvolvedor|    14998.33|   M|false|   2025-01-21|
|        19| Cliente 19|   39|Feira de Santana|    BA|   Engenheiro|    14974.64|   M| true|   2024-08-12|
|       159|Cliente 159|   35|       São Paulo|    SP|Desenvolvedor|     14967.5|   M|false|   2024-02-16|
|        30| Cliente 30|   56|        Salvador|    BA|    Professor|     14830.7|   F|false|   2024-05-04|
|       255|Cliente 255|   49|Feira de Santana|    BA|Desenvolvedor|    14822.31|   F|false|   2023-10-05|
|       187|Cliente 187|   46|        Salvador|    BA|    Professor|    14726.64|   F|false|   2023-03-28|
|       229|Cliente 229|   27|       

In [16]:
df_ordenado.explain(True)

== Parsed Logical Plan ==
'Sort ['renda_mensal DESC NULLS LAST], true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Sort [renda_mensal#23 DESC NULLS LAST], true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Sort [renda_mensal#23 DESC NULLS LAST], true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [renda_mensal#23 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(renda_mensal#23 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=66]
     

In [17]:
df_limitado = (
    df.limit(10)
)

In [18]:
df_limitado.explain(True)

== Parsed Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
GlobalLimit 10
+- LocalLimit 10
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
CollectLimit 10
+- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 pat

In [19]:
df_top10 = (
    df.orderBy(
        F.col("renda_mensal").desc()
    ).limit(10)
)

In [20]:
df_top10.explain(True)

== Parsed Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
TakeOrderedAndProject(limit=10, orderBy=[renda_mensal#23 DESC NUL

In [22]:
spark.stop()